# Iram, Garcia et al. (2024): the sequencing claims, from GEO

**Paper.** Iram T, Garcia MA, et al. (2024). SRF transcriptionally regulates the oligodendrocyte
cytoskeleton during CNS myelination. *PNAS* 121(12), e2307250121,
[doi:10.1073/pnas.2307250121](https://doi.org/10.1073/pnas.2307250121). Claims file:
`claims/001350/10.1073-pnas.2307250121.yaml`.

**Data.** The paper's TEM images are dandiset 001350 (version 0.250511.1527), tested in
`reproduce_iram_2024.ipynb`. Its sequencing data are not in the dandiset; the paper deposited them
in NCBI GEO, SuperSeries [GSE241561](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE241561)
(the claims file marks these claims `data: elsewhere`). This notebook reads the processed files
GEO holds: the nf-core ChIP-seq outputs of GSE241559 and the RSEM gene results of GSE241560.
GEO files are not versioned; the SHA-256 of every file read is printed below. Gene names come
from the Ensembl release 102 GTF (GRCm38) and peak sequences from the UCSC rn6 genome (Rnor 6.0,
the paper's assembly). The paper's Datasets S1 to S3 (its peak lists by group and its DESeq2 tables) are read from Europe
PMC's archive of the article's supplementary files (PMC10962977), only to check which reading of a
method the paper used; no result below is computed from them. Helpers are in `srf_geo.py`.

| Claim | Status |
|---|---|
| fig3a-chip-peaks | tested |
| fig3bc-chip-tss-carg | tested |
| fig3d-shared-carg-targets | tested |
| no-myelin-genes-direct-targets | tested |
| fig4ab-rnaseq-degs | tested |
| fig4cd-rnaseq-srf-actb-down | tested |
| figs4ef-assembly-not-disassembly-genes | tested |
| figs5ab-gsea-bulk | blocked: not runnable (GeneTrail 3 web service) |
| fig6b-snrnaseq-degs, fig6cd-ol-genes-depleted, fig6eg-disease-genes-up | blocked: not runnable (R pipeline: SoupX, DoubletFinder, Seurat, MAST) |

In [ ]:
import shutil
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats

import srf_geo

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()

consensus = srf_geo.read_bed("GSE241559_SRF.consensus_peaks.bed.gz")
igg_consensus = srf_geo.read_bed("GSE241559_IgG.consensus_peaks.bed.gz")
annotation = srf_geo.read_annotation()
replicate_peaks = {s: srf_geo.read_macs_xls(s) for g in srf_geo.CHIP_REPLICATES.values() for s in g}
igg_peaks = {g: srf_geo.read_macs_xls(s) for g, s in srf_geo.CHIP_IGG.items()}
counts, tpm, lengths = srf_geo.rnaseq_tables()
names = srf_geo.mouse_gene_names()
paper_tables = srf_geo.paper_datasets()   # the paper's Datasets S1 to S3, for comparison only
print(f"read {len(srf_geo.fetched)} files in {time.time() - started:.0f} s; "
      f"requests refused and retried: {sum(srf_geo.retries.values())}")

## Data check

In [ ]:
print("Files read (SHA-256):")
for url, digest in srf_geo.fetched.items():
    print(f"  {url.split('/')[-1]:60s} {digest}")
samples = srf_geo.RNASEQ_SAMPLES
print("\nRNA-seq samples by stage and genotype:")
print(samples.groupby(["stage", "genotype"]).size().to_string())
lib = counts.sum().round(0)
print("\nExpected counts per sample (millions):", (lib / 1e6).round(2).to_dict())
print("Genes in the RSEM tables:", counts.shape[0], "| with a name in Ensembl 102:", int(counts.index.isin(names.index).sum()))
markers = {"Pdgfra": "OPC", "Cspg4": "OPC", "Mbp": "OL", "Plp1": "OL", "Mog": "OL", "Srf": "target"}
ids = {g: names[names == g].index[0] for g in markers}
print("\nTPM of stage markers and Srf, by sample:")
print(tpm.loc[list(ids.values())].set_axis(list(ids)).T.join(samples.set_index("name")[["stage", "genotype"]]).round(1).to_string())

The stage labels agree with the markers: Pdgfra and Cspg4 are higher in the samples labelled OPC,
and the myelin genes higher in those labelled OL. Srf is lower in every KO sample than in the
controls of its stage, as a knockout should be (shown above; its test is claim fig4cd). The
sample counts match the Figure 4A and 4B legends (OPC: Flox 4, KO 3; OL: 4 and 4).

In [ ]:
print("ChIP-seq: SRF consensus peaks", len(consensus), "| IgG consensus peaks", len(igg_consensus),
      "| HOMER annotation rows", len(annotation))
print("Peaks per replicate (MACS2, q < 0.05):", {s: len(p) for s, p in replicate_peaks.items()})
print("IgG peaks per control:", {g: len(p) for g, p in igg_peaks.items()})
presence = pd.DataFrame({s: srf_geo.overlaps(consensus, p) for s, p in replicate_peaks.items()}, index=consensus.name)
presence["IgG_OPC"] = srf_geo.overlaps(consensus, igg_peaks["OPC"])
presence["IgG_OLG"] = srf_geo.overlaps(consensus, igg_peaks["OLG"])
presence["IgG_consensus"] = srf_geo.overlaps(consensus, igg_consensus)
n_opc = presence[srf_geo.CHIP_REPLICATES["OPC"]].sum(1)
n_olg = presence[srf_geo.CHIP_REPLICATES["OLG"]].sum(1)
print("Consensus peaks overlapped by at least one replicate peak:", int(((n_opc + n_olg) > 0).sum()), "of", len(consensus))
print("Replicates holding each consensus peak, OPC (of 4):", n_opc.value_counts().sort_index().to_dict())
print("Replicates holding each consensus peak, OLG (of 3):", n_olg.value_counts().sort_index().to_dict())
print("Consensus peaks overlapping an IgG peak:", presence[["IgG_OPC", "IgG_OLG", "IgG_consensus"]].sum().to_dict())
print("Chromosomes of the consensus peaks:", consensus.chr.value_counts().to_dict())
print("Annotation peak ids match the BED:", set(annotation.name) == set(consensus.name))
print("R available here:", shutil.which("R") is not None)

Every one of the 848 consensus peaks is overlapped by at least one replicate's peaks, as the
Methods' definition ("peaks present in at least one replicate") requires. The consensus file merges
all seven SRF replicates (OPC and day-3 oligodendrocytes, OLG in the file names).

## fig3a-chip-peaks

> SRF ChIP-seq in rat OPCs and day-3 oligodendrocytes finds 848 significant binding sites,
> narrowed to 329 high-confidence targets.

Expected 848 +/- 84.8 and 329 +/- 32.9. The Results define the high-confidence set as peaks "that
appeared consistently across replicates within the same group and not in the IgG control". The
paper does not say how many replicates make "consistently", nor which IgG peaks are excluded, so
every reading is run: present in all replicates of a group (OPC 4 of 4 or OLG 3 of 3), in a
majority (3 of 4, 2 of 3), or in at least 2; with IgG taken as the IgG consensus or the group's own
IgG control. The count is of consensus peaks high-confidence in either group.

The paper's Dataset S1 lists its peaks by group ("OPC high", "Common", "OL high"); it is compared
with the consensus file below, to see which peaks the paper treated as each group's.

In [ ]:
print("Significant SRF binding sites (consensus peaks):", len(consensus))
rows = []
for rule, (k_opc, k_olg) in {"all replicates": (4, 3), "majority": (3, 2), "at least 2": (2, 2)}.items():
    for igg in ["IgG consensus", "own group's IgG"]:
        no_igg_opc = ~presence["IgG_consensus"] if igg == "IgG consensus" else ~presence["IgG_OPC"]
        no_igg_olg = ~presence["IgG_consensus"] if igg == "IgG consensus" else ~presence["IgG_OLG"]
        hc_opc = (n_opc >= k_opc) & no_igg_opc
        hc_olg = (n_olg >= k_olg) & no_igg_olg
        rows.append({"replicates": rule, "IgG": igg, "OPC": int(hc_opc.sum()), "OLG": int(hc_olg.sum()),
                     "either group": int((hc_opc | hc_olg).sum()), "both groups": int((hc_opc & hc_olg).sum()),
                     "within 329 +/- 32.9": abs((hc_opc | hc_olg).sum() - 329) <= 32.9})
    # Variant: no IgG filter, which the paper's "and not in the IgG control" excludes.
    n_no_igg = int(((n_opc >= k_opc) | (n_olg >= k_olg)).sum())
    rows.append({"replicates": rule, "IgG": "none (variant)", "OPC": int((n_opc >= k_opc).sum()),
                 "OLG": int((n_olg >= k_olg).sum()), "either group": n_no_igg,
                 "both groups": int(((n_opc >= k_opc) & (n_olg >= k_olg)).sum()),
                 "within 329 +/- 32.9": abs(n_no_igg - 329) <= 32.9})
hc = pd.DataFrame(rows)
print(hc.to_string(index=False))
high_conf = ((n_opc >= 4) & ~presence["IgG_consensus"]) | ((n_olg >= 3) & ~presence["IgG_consensus"])
high_conf_majority = ((n_opc >= 3) & ~presence["IgG_consensus"]) | ((n_olg >= 2) & ~presence["IgG_consensus"])

# The paper's Dataset S1: its peak lists by group, by the consensus peak ids.
s1_group = pd.Series("not in Dataset S1", index=consensus.name)
for sheet, df in paper_tables["sd01"].items():
    s1_group[s1_group.index.isin(df.PeakID)] = sheet.strip()
print("\nDataset S1 sheets (peaks):", {k.strip(): len(v) for k, v in paper_tables["sd01"].items()},
      "| peaks listed in all:", int((s1_group != "not in Dataset S1").sum()))
igg_any = presence[["IgG_consensus", "IgG_OPC", "IgG_OLG"]].any(axis=1)
print("Dataset S1 sheet against the consensus file's replicates and IgG overlap:")
print(pd.crosstab([(n_opc > 0).rename("in an OPC replicate"), (n_olg > 0).rename("in an OLG replicate"),
                   igg_any.rename("overlaps an IgG peak")], s1_group).to_string())

**Result: partially reproduced.** The SRF consensus file holds 848 peaks, the paper's number of
significant binding sites; this is a count of the deposited file, not a re-analysis of the reads. The
high-confidence count depends on how "consistently across replicates" is read. With IgG peaks left
out, as the paper says, all replicates of a group gives 204 or 206 peaks, a majority 312 or 316
(inside 329 +/- 32.9), and at least two 404 or 409. A majority without the IgG filter gives 329,
but the paper excludes peaks in the IgG control. The paper's Dataset S1 does not settle it: its
three sheets hold 800 peaks, nearly the peaks present in at least one replicate of a group and in no
IgG control (799 of the 800 are; 15 such peaks are left out, and one listed peak overlaps an IgG
peak), so it lists no set of 329. The
848 holds; the 329 holds under one of three replicate rules, and the most literal reading of
"consistently", all replicates, gives 204.

## fig3bc-chip-tss-carg

> Most SRF peaks lie within 2 kb of the transcription start site, and the peak sequences are
> enriched for the SRF CArG motif.

The distance to the TSS is HOMER's annotation, which the paper used ("HOMER ... was then used to
annotate peaks with information from the closest known genomic feature"). "Most" is read as more
than half. The motif part is tested on the paper's window, -250 to +250 nt around each peak (taken
around the peak centre), for the SRF CArG motif under three readings: FIMO matches of the JASPAR SRF
matrix (MA0083.3) at FIMO's default p < 1e-4, the paper's tool; the consensus CC(A/T)6GG; and the
consensus with one mismatch. Two backgrounds: windows of the same width 3 kb either side, and the
peak sequences shuffled. MEME-ChIP's motif discovery, which found the paper's motif, is not run.

The one-mismatch consensus is this attempt's own looser reading, not one the paper names. Peak
windows are GC-richer than flank windows, which could bias the flank comparison; the shuffled
background keeps each peak's composition, and the flank comparison is also made within GC strata.

In [ ]:
dist = annotation.set_index("name")["Distance to TSS"].reindex(consensus.name)
within = dist.abs() <= 2000
print(f"All 848 peaks: within 2 kb of a TSS {int(within.sum())} ({100 * within.mean():.1f}%); within 1 kb "
      f"{100 * (dist.abs() <= 1000).mean():.1f}%; median distance {dist.abs().median():.0f} bp")
for label, sel in {"OPC peaks (in at least 1 OPC replicate)": n_opc > 0, "OLG peaks": n_olg > 0,
                   "high-confidence (all replicates, no IgG)": high_conf.to_numpy(),
                   "high-confidence (majority, no IgG)": high_conf_majority.to_numpy()}.items():
    print(f"{label}: {int(sel.sum())} peaks, within 2 kb {100 * within[sel].mean():.1f}%")
print("\nHOMER annotation classes:")
print(annotation.Annotation.str.split("(").str[0].str.strip().value_counts().to_string())

In [ ]:
genome = srf_geo.TwoBit()
consensus["ucsc"] = consensus.chr.map(srf_geo.ucsc_chrom)
consensus["centre"] = (consensus.start + consensus.end) // 2
on_main = consensus.ucsc.notna()
print(f"Peaks on assembled chromosomes: {int(on_main.sum())} of {len(consensus)} (the rest are on unplaced scaffolds, left out)")
W, F = srf_geo.MOTIF_HALF_WIDTH, srf_geo.FLANK_OFFSET

def windows(row):
    seq = genome.sequence(row.ucsc, row.centre - F - W, row.centre + F + W)
    return seq[F:F + 2 * W], seq[:2 * W], seq[-2 * W:]

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    seqs = list(ex.map(windows, consensus[on_main].itertuples()))
print(f"read {len(seqs)} sequences of {2 * (F + W)} bp in {time.time() - t0:.0f} s")
rng = np.random.default_rng(srf_geo.SHUFFLE_SEED)
shuffled = [srf_geo.shuffle(s[0], rng) for s in seqs]
background = srf_geo.base_composition([s[0] for s in seqs])
meme = srf_geo.srf_motif_meme(background)
print("FIMO background (peak windows, strand-symmetric):", {k: round(v, 3) for k, v in background.items()})
motif = pd.DataFrame(index=consensus[on_main].name)
READINGS = {"fimo": "FIMO MA0083.3 p<1e-4", 0: "consensus, strict", 1: "consensus, 1 mismatch"}
for key in READINGS:
    if key == "fimo":
        count = lambda xs: srf_geo.fimo_sites(xs, meme)
    else:
        count = lambda xs, k=key: np.array([srf_geo.carg_sites(x, k) for x in xs])
    motif[f"peak_{key}"] = count([s[0] for s in seqs])
    motif[f"flank_{key}"] = (count([s[1] for s in seqs]) > 0).astype(int) + (count([s[2] for s in seqs]) > 0).astype(int)
    motif[f"shuffled_{key}"] = count(shuffled)
gc = np.array([sum(c in "GC" for c in s[0].upper()) / max(1, sum(c in "ACGT" for c in s[0].upper())) for s in seqs])
gc_f = np.array([sum(c in "GC" for c in (s[1] + s[2]).upper()) / max(1, sum(c in "ACGT" for c in (s[1] + s[2]).upper())) for s in seqs])
print(f"GC content: peak windows {gc.mean():.3f}, flank windows {gc_f.mean():.3f}")
groups = {"all peaks": np.ones(len(motif), bool), "OPC peaks": (n_opc > 0).to_numpy()[on_main.to_numpy()],
          "OLG peaks": (n_olg > 0).to_numpy()[on_main.to_numpy()]}
rows = []
for key, label in READINGS.items():
    for gname, sel in groups.items():
        m = motif[sel]
        n = len(m)
        a = int((m[f"peak_{key}"] > 0).sum())
        fl = int(m[f"flank_{key}"].sum())          # flank windows with a site, of 2n
        sh = int((m[f"shuffled_{key}"] > 0).sum())
        rows.append({"CArG": label, "peaks": gname, "n": n,
                     "peak windows with site %": 100 * a / n, "flank windows %": 100 * fl / (2 * n),
                     "shuffled %": 100 * sh / n,
                     "odds vs flank": stats.fisher_exact([[a, n - a], [fl, 2 * n - fl]])[0],
                     "p vs flank": stats.fisher_exact([[a, n - a], [fl, 2 * n - fl]])[1],
                     "p vs shuffled": stats.fisher_exact([[a, n - a], [sh, n - sh]])[1]})
carg_table = pd.DataFrame(rows)
print(carg_table.round(4).to_string(index=False))

In [ ]:
# Peak windows are GC-richer than flank windows. The shuffled background keeps each peak's own
# base composition; here the flank comparison is also made within GC strata (quintiles of GC over
# peak and flank windows together), with the Mantel-Haenszel odds ratio over strata.
win_gc = np.concatenate([gc, [sum(c in "GC" for c in s[1].upper()) / max(1, sum(c in "ACGT" for c in s[1].upper())) for s in seqs],
                         [sum(c in "GC" for c in s[2].upper()) / max(1, sum(c in "ACGT" for c in s[2].upper())) for s in seqs]])
edges = np.quantile(win_gc, [0.2, 0.4, 0.6, 0.8])
stratum = np.digitize(win_gc, edges)
n = len(seqs)
is_peak = np.r_[np.ones(n, bool), np.zeros(2 * n, bool)]
for key, label_ in READINGS.items():
    fn = (lambda xs: srf_geo.fimo_sites(xs, meme)) if key == "fimo" else (lambda xs, k=key: np.array([srf_geo.carg_sites(x, k) for x in xs]))
    has = np.r_[motif[f"peak_{key}"].to_numpy() > 0, fn([s[1] for s in seqs]) > 0, fn([s[2] for s in seqs]) > 0]
    num = den = 0.0
    parts = []
    for q in range(5):
        sel = stratum == q
        a = int((has & is_peak & sel).sum()); b = int((~has & is_peak & sel).sum())
        c = int((has & ~is_peak & sel).sum()); d = int((~has & ~is_peak & sel).sum())
        t = a + b + c + d
        num += a * d / t; den += b * c / t
        parts.append(f"GC q{q + 1}: peaks {100 * a / max(1, a + b):.1f}% of {a + b}, flanks {100 * c / max(1, c + d):.1f}% of {c + d}")
    print(f"{label_}: Mantel-Haenszel odds ratio over GC strata {num / den:.2f}")
    print("   " + "; ".join(parts))

**Result: reproduced, resting on an assumption.** 545 of the 848 peaks (64.3%) are within 2 kb
of a TSS, and 76.0% and 76.3% of the high-confidence peaks. FIMO matches of the JASPAR SRF matrix
are in 10.7% of the peak windows, against 5.2% of the flank windows (odds 2.2, p < 0.0001) and 3.7%
of the shuffled peaks; the strict consensus gives 9.8% against 3.0% and 2.4%. Within GC strata the
enrichment over flanks stays (Mantel-Haenszel odds ratio 2.14 for FIMO and 2.75 for the strict
consensus), so the GC difference between peaks and flanks does not explain it. The one-mismatch
consensus, a looser reading of this attempt's own, is not enriched over all peaks (38.9% against
36.7%, p = 0.28), only in OLG peaks (p = 0.014). The outcome rests on the JASPAR matrix standing in
for the motif MEME-ChIP found, which is not run.

## fig3d-shared-carg-targets

> About half (60) of the CArG-containing target genes are shared between OPCs and
> oligodendrocytes, including immediate early genes and cytoskeletal genes such as Actb.

Expected 60 +/- 6. A gene is a target of a group when one of its peaks (HOMER's nearest gene) is a
peak of that group and carries a CArG box in the -250 to +250 window. The group's peaks are read two
ways: those present in at least one replicate of the group (the Methods' consensus per cell type)
or the high-confidence peaks of fig3a (all replicates, or a majority, with no IgG peak); and
the CArG box under the three readings of fig3bc.

In [ ]:
gene_of = annotation.set_index("name")["Gene Name"].reindex(motif.index)
opc_any = (n_opc > 0).to_numpy()[on_main.to_numpy()]
olg_any = (n_olg > 0).to_numpy()[on_main.to_numpy()]
opc_hc = ((n_opc >= 4) & ~presence["IgG_consensus"]).to_numpy()[on_main.to_numpy()]
olg_hc = ((n_olg >= 3) & ~presence["IgG_consensus"]).to_numpy()[on_main.to_numpy()]
opc_mj = ((n_opc >= 3) & ~presence["IgG_consensus"]).to_numpy()[on_main.to_numpy()]
olg_mj = ((n_olg >= 2) & ~presence["IgG_consensus"]).to_numpy()[on_main.to_numpy()]
# The paper's own peak lists (Dataset S1): OPC = "OPC high" and "Common", OL = "OL high" and "Common".
s1_on_main = s1_group.reindex(motif.index)
s1_opc = s1_on_main.isin(["OPC high peaks", "Common peaks"]).to_numpy()
s1_olg = s1_on_main.isin(["OL high peaks", "Common peaks"]).to_numpy()
rows, shared_sets = [], {}
for mm in READINGS:
    has = (motif[f"peak_{mm}"] > 0).to_numpy()
    for pname, (so, sl) in {"at least 1 replicate": (opc_any, olg_any), "high-confidence, all replicates": (opc_hc, olg_hc),
                                   "high-confidence, majority": (opc_mj, olg_mj),
                                   "Dataset S1 groups": (s1_opc, s1_olg)}.items():
        g_opc = set(gene_of[has & so].dropna())
        g_olg = set(gene_of[has & sl].dropna())
        shared = g_opc & g_olg
        shared_sets[(mm, pname)] = shared
        rows.append({"CArG": READINGS[mm], "peaks": pname, "OPC genes": len(g_opc),
                     "OLG genes": len(g_olg), "shared": len(shared), "union": len(g_opc | g_olg),
                     "shared % of union": 100 * len(shared) / max(1, len(g_opc | g_olg)),
                     "within 60 +/- 6": abs(len(shared) - 60) <= 6})
fig3d = pd.DataFrame(rows)
print(fig3d.round(1).to_string(index=False))
named = ["Egr1", "Egr2", "Egr3", "Fos", "Junb", "Srf", "Actb", "Actg1", "Arc", "Vcl"]
for key, shared in shared_sets.items():
    print(key, "named genes among shared:", [g for g in named if g in shared])
print("\nNamed genes with any SRF peak: peak, sites (FIMO, strict, 1 mismatch), OPC and OLG replicates:")
for g in named:
    ids = gene_of[gene_of == g].index
    print(g, [(i, int(motif.loc[i, "peak_fimo"]), int(motif.loc[i, "peak_0"]), int(motif.loc[i, "peak_1"]),
               int(n_opc[i]), int(n_olg[i])) for i in ids])

**Result: partially reproduced, resting on an assumption.** With FIMO matches and the Methods'
peaks per cell type (present in at least one replicate), 43 CArG target genes are shared between
OPC and OLG, 49.4% of the 87 in either group, so "about half" holds but the count is outside 60 +/-
6. The paper's own peak lists by group (Dataset S1) give 42 shared of 85. The named genes Egr1, Egr2,
Egr3, Fos, Junb, Srf, Actb, Actg1 and Vcl are all among the shared genes; Arc has no OPC peak. Of the
twelve readings, one gives a count inside the tolerance (56): the one-mismatch consensus, a reading
of this attempt's own, on the majority high-confidence peaks, and it drops Fos, Junb, Srf and Vcl
from the shared genes. Genes are HOMER's nearest genes, the paper's annotation; the paper's motif
from MEME-ChIP is not used.

## no-myelin-genes-direct-targets

> Classical myelin genes are not among the direct SRF targets found by ChIP-seq.

The paper names no list. The structural myelin genes Mbp, Plp1, Mog, Mag, Mobp, Cnp, Cldn11,
Pmp22 and Mal, with Opalin and the myelin regulator Myrf, are looked for among the nearest genes of
the SRF peaks (HOMER's annotation), over all 848 peaks, the high-confidence peaks and the peaks
with a CArG box.

In [ ]:
myelin = ["Mbp", "Plp1", "Mog", "Mag", "Mobp", "Cnp", "Cldn11", "Pmp22", "Mal", "Opalin", "Myrf"]
ann = annotation.set_index("name").reindex(consensus.name)
hits = ann[ann["Gene Name"].isin(myelin)]
print("Peaks whose nearest gene is a myelin gene (all 848):")
print(hits[["Gene Name", "Distance to TSS", "Annotation"]].assign(
    OPC_reps=n_opc[hits.index], OLG_reps=n_olg[hits.index], high_conf=high_conf[hits.index]).to_string())
carg_any = set(motif.index[(motif.peak_1 > 0) | (motif.peak_fimo > 0)])
print("\nOf these, with a CArG box (FIMO or 1 mismatch):", [i for i in hits.index if i in carg_any])
print("Of these, high-confidence:", list(hits.index[high_conf[hits.index].to_numpy()]))
print(f"Distinct nearest genes over all peaks: {ann['Gene Name'].nunique()}")

**Result: reproduced, resting on an assumption.** None of the eleven myelin genes is the nearest
gene of any of the 848 SRF peaks, so none is a target under any reading of the peak set or the
motif.

## fig4ab-rnaseq-degs

> SRF knockout changes the expression of 157 genes in cultured OPCs and 264 in day-3
> oligodendrocytes (adjusted p < 0.05).

Expected 157 +/- 15.7 and 264 +/- 26.4. DESeq2 "with standard settings" on raw gene counts from
RSEM (SI Methods), run with pydeseq2. The paper's "normalization for transcript length and
sequencing depth" and whether the stages were modelled apart are not settled, so three readings
are run: (a) each stage alone, expected counts rounded; (b) each stage alone, counts scaled from TPM
by gene length (tximport's lengthScaledTPM), which builds the length correction into the counts;
(c) one model of all 15 samples with a factor for stage and genotype, counts as in (a).

The standard way to give DESeq2 transcript lengths from RSEM output is tximport's average-length
offsets. pydeseq2 takes one size factor per sample and cannot take a gene-by-sample normalization
matrix, and R is not available, so that reading is not run; reading (b) is the length-scaled
alternative tximport offers. The paper's own DESeq2 tables (Datasets S2 and S3) are compared with
each reading.

In [ ]:
scaled = srf_geo.length_scaled_counts(counts, tpm, lengths)

results = {}
t0 = time.time()
for stage in ["OPC", "OL"]:
    cols = samples[samples.stage == stage].name.tolist()
    results[("a", stage)] = srf_geo.deseq(counts[cols], samples)
    results[("b", stage)] = srf_geo.deseq(scaled[cols], samples)
joint = samples.assign(group=samples.stage + "_" + samples.genotype)
for stage in ["OPC", "OL"]:
    results[("c", stage)] = srf_geo.deseq(counts[samples.name.tolist()], joint, design="~group",
                                          contrast=("group", f"{stage}_KO", f"{stage}_Flox"))
print(f"{len(results)} DESeq2 fits in {time.time() - t0:.0f} s")
rows = []
for (reading, stage), r in results.items():
    sig = r[r.padj < 0.05]
    exp, tol = (157, 15.7) if stage == "OPC" else (264, 26.4)
    rows.append({"reading": reading, "stage": stage, "genes tested": int(r.padj.notna().sum()),
                 "padj < 0.05": len(sig), "up": int((sig.log2FoldChange > 0).sum()),
                 "down": int((sig.log2FoldChange < 0).sum()), "expected": exp,
                 "within tolerance": abs(len(sig) - exp) <= tol, "p < 0.05": int((r.pvalue < 0.05).sum())})
degs = pd.DataFrame(rows)
print(degs.to_string(index=False))

In [ ]:
# The paper's own DESeq2 tables (Dataset S2, OPCs; Dataset S3, oligodendrocytes), against each reading.
paper_stats = {"OPC": paper_tables["sd02"]["OPC_stats"], "OL": paper_tables["sd03"]["OL_stats"]}
rows = []
for stage, ps in paper_stats.items():
    ps = ps.dropna(subset=["gene"]).drop_duplicates("gene").set_index("gene")
    p_sig = set(ps.index[ps.padj < 0.05])
    for reading in ["a", "b", "c"]:
        r = results[(reading, stage)].dropna(subset=["gene"]).drop_duplicates("gene").set_index("gene")
        common = ps.index.intersection(r.index)
        sig = set(r.index[r.padj < 0.05])
        rows.append({"stage": stage, "reading": reading, "paper padj < 0.05": len(p_sig), "this padj < 0.05": len(sig),
                     "both": len(p_sig & sig), "Jaccard": round(len(p_sig & sig) / len(p_sig | sig), 3),
                     "log2FC Spearman": round(stats.spearmanr(ps.loc[common, "log2FoldChange"], r.loc[common, "log2FoldChange"],
                                                              nan_policy="omit")[0], 3),
                     "median baseMean ratio (this / paper)": round(float((r.loc[common, "baseMean"] / ps.loc[common, "baseMean"]).median()), 3),
                     "baseMean Spearman": round(stats.spearmanr(ps.loc[common, "baseMean"], r.loc[common, "baseMean"])[0], 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe paper's tables for Srf, Actb, Actg1 and Cfl1:")
for stage, ps in paper_stats.items():
    print(stage); print(ps[ps.gene.isin(["Srf", "Actb", "Actg1", "Cfl1"])].round(5).to_string(index=False))

**Result: reproduced, resting on an assumption.** With counts that carry the length correction
the Methods describe (reading b), 160 genes have padj < 0.05 in OPCs and 264 in oligodendrocytes,
inside 157 +/- 15.7 and 264 +/- 26.4. Rounded expected counts (reading a) give 117 and 294, and the
joint model (reading c) 130 and 423, outside the tolerance. The paper's own tables (Datasets S2 and
S3) hold 154 and 261 genes with padj < 0.05, and their baseMean values are on the scale of the
expected counts (median ratio of reading a to the paper 0.99 and 0.99), not of length-scaled counts
(0.73 and 0.85). That is what tximport's offsets give, so the paper most likely used offsets and not
reading (b). Reading (b) is the closest of the three to the paper's tables in fold changes (Spearman
0.88 and 0.94, against 0.82 and 0.85 for a) and, in oligodendrocytes, in the genes called (Jaccard
0.76, against 0.50); in OPCs reading (a) shares slightly more genes with the paper (Jaccard 0.56
against 0.46). The counts match under the stand-in for the length correction, not under the
paper's exact procedure, which is not run.

## fig4cd-rnaseq-srf-actb-down

> Srf and Actb are down-regulated in SRF-KO OPCs and oligodendrocytes in culture, and actin genes
> among the CArG-containing direct targets are down-regulated.

Down-regulated is read as a negative log2 fold change with DESeq2 padj < 0.05 (the paper's
threshold for differential expression; Figure 4C and 4D, "Statistics by DESeq2"). For the second
part, Figure 4H plots CArG-containing direct targets "significantly altered ... (P val < 0.05)";
the actin genes (Act*) among the genes with a CArG peak are listed with their change.

In [ ]:
def gene_rows(genes):
    out = []
    for (reading, stage), r in results.items():
        for g in genes:
            x = r[r.gene == g]
            if len(x):
                x = x.iloc[0]
                out.append({"reading": reading, "stage": stage, "gene": g, "baseMean": x.baseMean,
                            "log2FC": x.log2FoldChange, "pvalue": x.pvalue, "padj": x.padj})
    return pd.DataFrame(out)

sa = gene_rows(["Srf", "Actb"])
print(sa.round(5).to_string(index=False))
print("\nSrf and Actb down with padj < 0.05, by reading and stage:")
print(sa.assign(down=(sa.log2FC < 0) & (sa.padj < 0.05)).pivot_table(index=["reading", "stage"], columns="gene", values="down").to_string())
carg_genes = set(gene_of[((motif.peak_fimo > 0) | (motif.peak_1 > 0)).to_numpy()].dropna())
actin = sorted(g for g in carg_genes if g.startswith("Act") and len(g) <= 5)
print("\nActin genes (Act*) with a CArG peak (FIMO or 1 mismatch):", actin)
print("Their peaks' sites (FIMO, strict, 1 mismatch):", {g: [(int(motif.loc[i, 'peak_fimo']), int(motif.loc[i, 'peak_0']), int(motif.loc[i, 'peak_1'])) for i in gene_of[gene_of == g].index] for g in actin})
act = gene_rows(actin)
print(act[act.pvalue < 0.05].round(5).to_string(index=False))

**Result: reproduced, resting on an assumption.** Under readings (a) and (b), Srf and Actb are
lower in KO with padj < 0.05 in both stages (reading b: Srf log2FC -4.82 in OPCs and -1.48 in
oligodendrocytes, Actb -0.41 and -1.43). Under the joint model (c), Actb in OPCs is not (padj
0.51). Actb and Actg1 are the actin genes with a CArG peak; where they change at p < 0.05 (Actb in
both stages, Actg1 in oligodendrocytes) they are lower in KO, and neither is higher.
The paper's own tables agree with the per-stage readings: Actb log2FC -0.34, padj 0.044 in OPCs and
-1.33, padj 0.00001 in oligodendrocytes, and Dataset S2 holds the OPC samples alone, as a per-stage
model would.

## figs4ef-assembly-not-disassembly-genes

> In cultured OPCs and oligodendrocytes, SRF mainly regulates genes that promote actin assembly,
> actin itself included, and the main actin disassembly genes are largely unchanged in SRF-KO
> cells. Cfl1 is the exception, lower in SRF-KO oligodendrocytes and unchanged in OPCs.

The claims file was amended after a reader's answer (questions/001350.yaml); it earlier named Cfl1
as an unregulated disassembly gene. The disassembly genes are the 13 of Fig. S4E. A gene counts as
changed when DESeq2 padj < 0.05, the paper's threshold for differential expression (Results, "157 differentially
expressed genes ( P .adj < 0.05)"; Figure 4A legend). The parts tested: the 12 disassembly genes other than Cfl1 unchanged (at most one
changed in a stage is read as "largely"; the count is printed); Cfl1 lower in KO oligodendrocytes
and unchanged in OPCs; the actin genes Actb and Actg1 lower in KO. Fig. S4F prints p 0.007 for Cfl1
in oligodendrocytes (DESeq2, 4 and 4 samples); the DESeq2 p and padj here are printed beside it.

In [ ]:
dis = gene_rows(srf_geo.DISASSEMBLY_GENES)
print(dis[dis.reading == "a"].round(4).to_string(index=False))
summary_rows = []
for (reading, stage), grp in dis.groupby(["reading", "stage"]):
    s = grp[grp.padj < 0.05]
    other = grp[grp.gene != "Cfl1"]
    cfl1 = grp[grp.gene == "Cfl1"].iloc[0]
    summary_rows.append({"reading": reading, "stage": stage, "disassembly genes tested": int(grp.padj.notna().sum()),
                         "padj < 0.05": len(s), "which": ", ".join(f"{g} ({v:+.2f})" for g, v in zip(s.gene, s.log2FC)),
                         "others tested": int(other.padj.notna().sum()), "others padj < 0.05": int((other.padj < 0.05).sum()),
                         "Cfl1 log2FC": round(cfl1.log2FC, 3), "Cfl1 p": cfl1.pvalue, "Cfl1 padj": cfl1.padj,
                         "Cfl1 lower, padj < 0.05": bool(cfl1.log2FC < 0 and cfl1.padj < 0.05)})
print(pd.DataFrame(summary_rows).to_string(index=False))
print("\nCfl1 (Fig. S4F prints p 0.007 for oligodendrocytes, KO lower):")
print(dis[dis.gene == "Cfl1"].round(5).to_string(index=False))
print("\nDisassembly genes with p < 0.05 (unadjusted), reading a:")
print(dis[(dis.reading == "a") & (dis.pvalue < 0.05)].round(5).to_string(index=False))
print("\nActin genes Actb and Actg1:")
print(gene_rows(["Actb", "Actg1"]).round(5).to_string(index=False))
asm = gene_rows(srf_geo.ASSEMBLY_GENES)
chip_genes = set(annotation["Gene Name"].dropna())
print("\nVariant, actin assembly genes (the paper lists none; Arp2/3, WASP/WAVE, formins, profilins, Ena/VASP and actin):")
print("Missing from the counts:", sorted(set(srf_geo.ASSEMBLY_GENES) - set(asm.gene)))
print("With an SRF ChIP peak (HOMER nearest gene):", [g for g in srf_geo.ASSEMBLY_GENES if g in chip_genes])
print(asm[(asm.reading == "a") & (asm.padj < 0.05)].round(5).to_string(index=False))
asm_rows = []
for (reading, stage), grp in asm.groupby(["reading", "stage"]):
    s = grp[grp.padj < 0.05]
    d = dis[(dis.reading == reading) & (dis.stage == stage)]
    asm_rows.append({"reading": reading, "stage": stage, "assembly tested": int(grp.padj.notna().sum()),
                     "assembly padj < 0.05": len(s), "which": ", ".join(f"{g} ({v:+.2f})" for g, v in zip(s.gene, s.log2FC)),
                     "disassembly tested": int(d.padj.notna().sum()), "disassembly padj < 0.05": int((d.padj < 0.05).sum()),
                     "all genes padj < 0.05 %": round(100 * float((results[(reading, stage)].padj < 0.05).mean()), 2)})
print(pd.DataFrame(asm_rows).to_string(index=False))
frac = {k: float((v.padj < 0.05).mean()) for k, v in results.items()}
print("\nFraction of all tested genes with padj < 0.05, by reading and stage:", {f"{a} {b}": round(f, 4) for (a, b), f in frac.items()})
print(f"\nnotebook ran in {time.time() - started:.0f} s")

**Result: partially reproduced, resting on an assumption.** None of the 12 disassembly genes of Fig.
S4E other than Cfl1 has padj < 0.05 in oligodendrocytes, and none of the 12 or 13 tested in OPCs does
(Ssh3 is too low in OPCs under readings a and b), under every reading. Cfl1 is lower in KO
oligodendrocytes (log2FC -0.78, p 0.0003, padj 0.024 in reading a; padj 0.029 in b and 0.0016 in
c), where Fig. S4F prints p 0.007, and unchanged in OPCs (log2FC -0.05, padj 0.91); the paper's own
table gives padj 0.038 and 0.93. Actin itself (Actb) is lower in KO in both stages under readings (a)
and (b); under the joint model (c) it is not in OPCs (padj 0.51). The paper names no set of actin
assembly genes. Of 26 standard ones (Arp2/3 subunits, WASP and WAVE, formins, profilins, Ena/VASP,
and Actb and Actg1), Actb is the only one with padj < 0.05, in either stage and under every reading
but (c) in OPCs, and Actb and Actg1 are the only ones with an SRF peak. In oligodendrocytes one of
the 26 assembly genes changes and one of the 13 disassembly genes does, so the expression data
show the disassembly genes largely unchanged and actin itself regulated, but not that SRF mainly
regulates assembly genes beyond actin. The assembly set is this attempt's choice.

## Claims not runnable here

- figs5ab-gsea-bulk: the paper's gene set enrichment used the GeneTrail 3 web service with its
  defaults; it cannot be run in a locked environment.
- fig6b-snrnaseq-degs, fig6cd-ol-genes-depleted, fig6eg-disease-genes-up: the single-nucleus
  analysis is an R pipeline (SoupX, DoubletFinder, scTransform and Seurat integration and
  clustering with manual annotation, MAST), and the claims rest on its clusters 0 and 2 and its MAST
  statistics. R is not available here (printed in the data check) and these packages cannot be
  installed in the entry's locked Python environment. GEO holds, for each of the two pooled samples
  (one SRF-Flox, one SRF-cKO), the Cell Ranger count matrices, raw, filtered and after SoupX (listed
  below), but no cluster labels or cell annotations, so the paper's clusters 0 and 2 cannot be
  recovered from it without re-running the pipeline.

In [ ]:
# What GEO holds for the single-nucleus RNA-seq (sub-series GSE243678 of GSE241561).
for acc in ["GSE243678", "GSM7794023", "GSM7794024"]:
    for line in srf_geo.geo_record(acc):
        if line.startswith(("!Series_title", "!Series_supplementary_file", "!Series_sample_id", "!Sample_title",
                            "!Sample_supplementary_file")) or (line.startswith("!Sample_data_processing") and "Supplementary" in line):
            print(acc, line.split("/")[-1] if "_file" in line else line)

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig3a-chip-peaks | partially reproduced (848 peaks; 329 high-confidence only under a majority-of-replicates reading: 312 or 316, against 204 to 206 with all replicates) |
| fig3bc-chip-tss-carg | reproduced (64.3% within 2 kb; CArG enriched under FIMO and the strict consensus) |
| fig3d-shared-carg-targets | partially reproduced (43 shared genes, about half, against 60) |
| no-myelin-genes-direct-targets | reproduced |
| fig4ab-rnaseq-degs | reproduced (160 and 264 genes under the length-scaled reading) |
| fig4cd-rnaseq-srf-actb-down | reproduced |
| figs4ef-assembly-not-disassembly-genes | partially reproduced (disassembly genes unchanged but Cfl1, as the claim says; actin down, but no other assembly gene changes) |
| figs5ab-gsea-bulk | blocked: not runnable |
| fig6b-snrnaseq-degs, fig6cd-ol-genes-depleted, fig6eg-disease-genes-up | blocked: not runnable |

Outcomes marked as resting on an assumption are explained in the record. How the analysis differs
from the paper's: DESeq2 is run as pydeseq2, without tximport's length offsets (reading b uses
length-scaled counts in their place; the paper's own tables are on the expected-count scale, as offsets give); motif matches are FIMO with the JASPAR SRF matrix and counts of the
CArG consensus, without MEME-ChIP's motif discovery; gene names are from Ensembl release 102.

What was found about the data: GEO's processed files agree with the paper's description (848
consensus peaks; 4 and 3 OPC samples, 4 and 4 oligodendrocyte samples; Srf lower in every KO
sample). Fig. S4F shows Cfl1 lower in KO oligodendrocytes, as the data do, while the Results say
SRF does not regulate genes that regulate actin disassembly and do not name the exception. The
paper's DESeq2 tables (Datasets S2 and S3) hold 154 and 261 genes with padj < 0.05, where the text
gives 157 and 264. Dataset S1 lists 800 peaks by group, not the 329 high-confidence peaks.